In [1]:
#!pip install pymupdf langchain langchain-community chromadb langchain_huggingface sentence-transformers ollama

In [2]:
## Cell 1 — Imports
import pdfplumber
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
# from langchain_huggingface import HuggingFaceEmbeddings
import ollama

/home/michael/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Cell 2 — Load and extract text from PDF
def load_pdf(path: str) -> str:
    with pdfplumber.open(path) as pdf:
        return "\n".join(page.extract_text() for page in pdf.pages)

pdf_text = load_pdf("./data/causal_roadmap.pdf")  # <-- change to your PDF path
print(f"Loaded {len(pdf_text):,} characters")

Loaded 73,180 characters


In [4]:
# Cell 3 — Split into chunks and build vector store
splitter = RecursiveCharacterTextSplitter(chunk_size=2500, chunk_overlap=250)
chunks = splitter.split_text(pdf_text)
print(f"Created {len(chunks)} chunks")

embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
# embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5") # <- alternmative embeddings model
vectorstore = Chroma.from_texts(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
print("Vector store ready ✓")

Created 33 chunks


/tmp/ipykernel_20794/3816481435.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17908.03it/s]


Vector store ready ✓


In [5]:
# Cell 4 — Chat function
def chat(question: str, history: list[dict]) -> str:
    # Retrieve relevant chunks from the PDF
    docs = retriever.invoke(question)
    context = "\n\n---\n\n".join(d.page_content for d in docs)

    # Build the prompt
    system_prompt = (
        "You are a helpful assistant. Answer the user's question using ONLY "
        "the context below. If the answer is not in the context, say so.\n\n"
        f"CONTEXT:\n{context}"
    )

    # Append this turn to history
    history.append({"role": "user", "content": question})

    response = ollama.chat(
        model="qwen3.5:9b",   # or "gemma:2b", "gemma:7b", etc.
        messages=[{"role": "system", "content": system_prompt}] + history,
        options={"num_ctx": 32768} 
    )

    answer = response["message"]["content"]
    history.append({"role": "assistant", "content": answer})
    return answer

In [6]:
# Cell 5 — Interactive chat loop (run this cell to start chatting)
history = []
print("PDF Chatbot ready. Type 'quit' to exit.\n")

while True:
    question = input("You: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        break
    if not question:
        continue
    answer = chat(question, history)
    print(f"\nLLM: {answer}\n")

PDF Chatbot ready. Type 'quit' to exit.


LLM: Based on the provided context, the seven steps of the Roadmap for specifying key elements of a study design and analysis plan are summarized below:

**Step 1: Causal Question, Model, and Estimand**
*   **1a & 1b:** Specify the causal question and estimand (including ICH E9(R1) attributes like population, treatment, and endpoint) and specify the causal model based on background knowledge.
*   Document the study type (e.g., randomized trial, retrospective cohort), document censoring, competing risks, or intercurrent events, and adjust the question as needed.
*   Use causal models and graphs to describe causal pathways between intervention and outcome variables, including mediators and colliders.

**Step 2: Define the Observed Data**
*   Define the observed data to be or have been collected.
*   Document how inclusion/exclusion criteria, treatment variables, outcomes, and other relevant variables are measured.
*   Define time zero and documen

In [7]:
model="qwen3.5:9b"
! ollama stop {model}

]11;?\⠙ 